# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisal-0065/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


One row represents the **daily performance of one content item**. For this assignment, I will use the **March 2026** data as the development time window.


In [14]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Hugging Face token loaded:", HF_TOKEN is not None)


Hugging Face token loaded: True


In [15]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("Dataset access successful!")
    print("Dataset:", info.id)
except Exception as e:
    print("Dataset access failed:")
    print(e)

Dataset access successful!
Dataset: FlyRank/internship-warehouse


In [16]:
from huggingface_hub import hf_hub_download
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(file_path)

print("March 2026 shape:", march_df.shape)
print("\nColumns:")
print(march_df.columns.tolist())

display(march_df.head())

March 2026 shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
print("Rows:", len(march_df))
print("Date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

print("\nUnique content items:", march_df["content_hash_id"].nunique())
print("Unique clients:", march_df["client_hash_id"].nunique())

Rows: 9841378
Date range: 2026-03-01 to 2026-03-31

Unique content items: 331437
Unique clients: 55


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions`

**Label:** I will use a future performance change as the label, based on observed data from a later time window.

**Context:** `client_hash_id`, `content_hash_id`, and `report_date` identify the client, content item, and observation date.

**Excluded:** I will exclude future information from the features because it would not be available at the decision moment and could cause data leakage.


In [18]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("Dataset access successful!")
    print("Dataset:", info.id)
except Exception as e:
    print("Dataset access failed:")
    print(e)


Dataset access successful!
Dataset: FlyRank/internship-warehouse


In [19]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Files in warehouse:")
for file in files:
    print(file)

Files in warehouse:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_da

In [20]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

context_cols = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("Features:", feature_cols)
print("Context:", context_cols)
print("Excluded: future performance fields")

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']
Context: ['client_hash_id', 'content_hash_id', 'report_date']
Excluded: future performance fields


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
grain_check = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
)

print("Total rows:", len(march_df))
print("Unique date-client-content combinations:", len(grain_check))
print("Maximum rows per combination:", grain_check.max())


Total rows: 9841378
Unique date-client-content combinations: 9841378
Maximum rows per combination: 1


In [22]:
print("March 2026 row count:", len(march_df))
print("Earliest date:", march_df["report_date"].min())
print("Latest date:", march_df["report_date"].max())

March 2026 row count: 9841378
Earliest date: 2026-03-01
Latest date: 2026-03-31


In [23]:
available_gsc = march_df[march_df["gsc_data_available"].isna() == False]

print("Rows with GSC data available:", len(available_gsc))
print("Total rows:", len(march_df))

Rows with GSC data available: 9841378
Total rows: 9841378


In [24]:
available_gsc = march_df[march_df["gsc_data_available"].eq(True)]

print("Rows where gsc_data_available IS TRUE:", len(available_gsc))
print("Total rows:", len(march_df))

Rows where gsc_data_available IS TRUE: 3611061
Total rows: 9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



This data shows observed daily performance, but it cannot prove that one factor caused a change in performance. Some rows do not have GSC data available, so the available history is not balanced across all content. Results should be treated as directional decision-support rather than causal proof.
